In [1]:
import pandas as pd
df2 = pd.read_csv('../data/dataset-tickets-multi-lang-4-20k.csv')
print(df2.shape)
print(df2.columns.tolist())

(20000, 15)
['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']


In [2]:
df_en = df2[df2['language'] == 'en']
print(df_en.shape)

print(df_en['type'].value_counts())
print()
print(df_en['queue'].value_counts())

(11923, 15)
type
Incident    4642
Request     3498
Problem     2498
Change      1285
Name: count, dtype: int64

queue
Technical Support                  3412
Product Support                    2232
Customer Service                   1859
IT Support                         1391
Billing and Payments               1302
Returns and Exchanges               582
Service Outages and Maintenance     442
Sales and Pre-Sales                 330
Human Resources                     205
General Inquiry                     168
Name: count, dtype: int64


In [4]:
for cat in df_en['queue'].unique():
    print(f"\n=== {cat} ===")
    for text in df_en[df_en['queue'] == cat]['body'].sample(min(3, len(df_en[df_en['queue']==cat])), random_state=1):
        print("-", text[:150])


=== Customer Service ===
- Dear Customer Support, I am inquiring about the integration options for the Google Nest Wifi Router with our SaaS project management tools. Could you 
- Could you provide information on digital strategies for promoting brand growth? I'm interested in learning about the company's approaches to online ma
- We've noticed a substantial decline in engagement for our digital marketing campaigns, which might be related to compatibility issues with the recent 

=== Technical Support ===
- Hello Customer Support, I am writing to seek details about the digital tactics your firm employs for brand expansion and advancement. Could you kindly
- Respected Customer Support, I am contacting you to address a problem with the Jenkins build that has failed owing to a Git integration error. It is po
- Our team is in need of detailed guidance concerning the implementation and integration strategies for our project management tools. Could you share mo

=== IT Support ===
- Seeking

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

x=df_en['body']
y=df_en['queue']

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42,stratify=y)
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
clt=LogisticRegression(max_iter=1000,class_weight='balanced')
clt.fit(X_train_vec, y_train)
print(classification_report(y_test, clt.predict(X_test_vec)))

                                 precision    recall  f1-score   support

           Billing and Payments       0.70      0.69      0.69       261
               Customer Service       0.32      0.26      0.28       372
                General Inquiry       0.11      0.38      0.17        34
                Human Resources       0.24      0.61      0.34        41
                     IT Support       0.35      0.40      0.37       278
                Product Support       0.44      0.29      0.35       446
          Returns and Exchanges       0.22      0.44      0.29       116
            Sales and Pre-Sales       0.13      0.35      0.19        66
Service Outages and Maintenance       0.33      0.61      0.43        88
              Technical Support       0.54      0.34      0.41       683

                       accuracy                           0.38      2385
                      macro avg       0.34      0.44      0.35      2385
                   weighted avg       0.44      0

In [11]:
df_en['answer'].isna().sum()

np.int64(3)

In [8]:
df_en['body'].isna().sum()
df_en = df_en.dropna(subset=['body'])

In [12]:
kb = df_en.dropna(subset=['answer']).copy()
print(kb.shape)

(11919, 15)


In [14]:
from sentence_transformers import SentenceTransformer
embedder=SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\LENOVO\Documents\ai-support-triage\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LENOVO\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
kb_texts = kb['body'].tolist()
kb_embeddings = embedder.encode(kb_texts, show_progress_bar=True)

Batches:   0%|          | 0/373 [00:00<?, ?it/s]

In [16]:
print(kb_embeddings.shape)

(11919, 384)


In [17]:
import chromadb

client = chromadb.Client()
collection = client.create_collection(name="support_tickets")

In [20]:
print("we are making the database in chromadb and inserting the knowledge base into it")
print("we are also grouping similar tickets together based on their queue and storing that information in the metadata")
client.delete_collection(name="support_tickets")
collection = client.create_collection(name="support_tickets")

batch_size = 5000

for start in range(0, len(kb), batch_size):
    end = start + batch_size
    collection.add(
        embeddings=kb_embeddings[start:end].tolist(),
        documents=kb["answer"].tolist()[start:end],
        ids=[str(i) for i in range(start, min(end, len(kb)))],
        metadatas=[{'queue': q} for q in kb['queue']][start:end]
    )
    print(f"Inserted rows {start} to {end}")

Inserted rows 0 to 5000
Inserted rows 5000 to 10000
Inserted rows 10000 to 15000


In [21]:
print(collection.count())

11919


In [24]:
print("we are making a test ticket and seeing if it gives 3 similar tickets from the knowledge base")
new_ticket="My subscription was charged twice this month and i want a refund for the extra charge."
new_ticket_embedding = embedder.encode([new_ticket])
results = collection.query(
    query_embeddings=new_ticket_embedding.tolist(),
    n_results=3
)
print(results)


{'ids': [['9526', '11391', '7895']], 'embeddings': None, 'documents': [['We will investigate the duplicate charge issue and process the necessary refund. Please allow us to review your account information (<acc_num>) to resolve the matter promptly.', '<name>, we apologize for the issue with your subscription renewal payment. We understand you have already tried contacting support and reviewing your billing history. We would like to assist you and investigate the matter. Could you please provide your <acc_num> and the date of the duplicate charge? We will work on refunding the extra amount as soon as possible. Please call <tel_num> at your convenience to discuss further.', 'We will investigate the issue with the duplicate charges and process the necessary refund. Please allow us some time to review your account information (<acc_num>) and resolve the matter promptly.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'queue': 'Billing an

In [25]:
print("we are getting the api key from the .env file")
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENROUTER_API_KEY")

print(api_key is not None)

True


In [28]:
print("we are getting the free models from openrouter.ai")
import requests

response = requests.get("https://openrouter.ai/api/v1/models")
models = response.json()['data']

free_models = [m['id'] for m in models if m['pricing']['prompt'] == '0']
print(free_models)

we are getting the free models from openrouter.ai
['inclusionai/ling-3.0-flash-vl:free', 'nex-agi/nex-n2.5-mini:free', 'nex-agi/nex-n2.5-pro:free', 'inclusionai/ling-3.0-flash-sante:free', 'inclusionai/ling-3.0-flash-fin:free', 'dots-studio/dots-3-note-preview:free', 'liquid/lfm-2.5-2.6b:free', 'nvidia/nemotron-3.5-lightning:free', 'thinkingmachines/inkling-small:free', 'poolside/laguna-s-2.1:free', 'thinkingmachines/inkling:free', 'poolside/laguna-xs-2.1:free', 'cohere/north-mini-code:free', 'nvidia/nemotron-3.5-content-safety:free', 'nvidia/nemotron-3-ultra-550b-a55b:free', 'nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free', 'google/gemma-4-26b-a4b-it:free', 'google/gemma-4-31b-it:free', 'google/lyria-3-pro-preview', 'google/lyria-3-clip-preview', 'nvidia/nemotron-3-super-120b-a12b:free', 'openrouter/free']


In [31]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)
FREE_MODEL = "nvidia/nemotron-3-super-120b-a12b:free"
response = client.chat.completions.create(
    model=FREE_MODEL,
    messages=[
        {"role": "user", "content": "Say hello and confirm you're working."}
    ]
)

print(response.choices[0].message.content)


Hello! I'm here and working properly. How can I assist you today? 😊


In [32]:
print("the models often have rate limits and dont work but when retried do work we are writing a replay function")
import time

def call_llm_with_retry(model, messages, max_retries=3, wait_seconds=5):
    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=messages
            )
            return response.choices[0].message.content
        except Exception as e:
            print(f"Attempt {attempt} failed: {e}")
            if attempt < max_retries:
                print(f"Waiting {wait_seconds} seconds before retrying...")
                time.sleep(wait_seconds)
            else:
                print("Giving up after max retries.")
                raise

the models often have rate limits and dont work but when retried do work we are writing a replay function


In [34]:
reply = call_llm_with_retry(FREE_MODEL, [{"role": "user", "content": "Say hello and confirm you're working."}])
print(reply)

Hello! I’m here and ready to help. Let me know what you need!


In [35]:
def resolution_agent(new_ticket_text, n_results=3):
    query_embedding = embedder.encode([new_ticket_text])
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=n_results
    )
    retrieved_resolutions = results['documents'][0]

    context = "\n\n".join([f"Past resolution {i+1}: {r}" for i, r in enumerate(retrieved_resolutions)])

    prompt = f"""You are a customer support agent. A new ticket has come in:

"{new_ticket_text}"

Here are similar past resolutions for reference:

{context}

Write a helpful, professional reply to the new ticket, using the past resolutions as guidance. Do not just copy them — tailor the reply to this specific ticket."""

    reply = call_llm_with_retry(FREE_MODEL, [{"role": "user", "content": prompt}])
    return reply

In [36]:
draft = resolution_agent("My subscription was charged twice this month and I want a refund")
print(draft)

Hello,

I’m sorry to hear that you were charged twice for your subscription this month. I’d be happy to look into this and arrange a refund for the duplicate charge.

To get started, could you please provide your account number (or the email address associated with the subscription) and the dates of the two charges you see on your statement? Once I have that information, I’ll investigate the billing discrepancy and process the refund as quickly as possible.

Thank you for bringing this to our attention, and please let me know if there’s anything else I can help with.

Best regards,  
[Your Name]  
Customer Support Team  
[Company Name]  
[Phone Number] | [Support Email]


In [37]:
def escalation_agent(new_ticket_text, priority, draft_reply):
    prompt = f"""You are reviewing a customer support ticket and a draft reply before it gets sent.

Ticket: "{new_ticket_text}"
Priority: {priority}
Draft reply: "{draft_reply}"

Decide whether this reply should be sent automatically, or escalated to a human agent for review. Consider: is the priority high/critical, does the draft reply seem confident and complete, or does it seem uncertain or risky to send without human review?

Respond in this exact format:
Decision: [AUTO-SEND or ESCALATE]
Reason: [one sentence explaining why]"""

    decision = call_llm_with_retry(FREE_MODEL, [{"role": "user", "content": prompt}])
    return decision

In [38]:
decision = escalation_agent(
    "My subscription was charged twice this month and I want a refund",
    "high",
    draft
)
print(decision)

Decision: AUTO-SEND
Reason: The reply courteously acknowledges the double charge, clearly requests the necessary information to investigate, and offers a refund, making it suitable for automatic sending despite the high priority.
